### Next Word Generator

- LLMs in general are trained on huge text corpus. 
- Preprocessing of the corpus involves removal of punctuations, stopwords etc and creating a vocabulary. 
- Here for mimicking the working of llm we are using 25 short sentences for which preprocessing is not required. 

In [1]:
sentences = [
    "I am learning machine learning",
    "I like deep learning models",
    "I am playing with friends",
    "I am reading a book",
    "I am studying mathematics",
    "I am working on a project",
    "machine learning is fun to learn",
    "deep learning uses neural networks",
    "natural language processing is interesting",
    "python is widely used in data science",
    "data science includes machine learning",
    "students are studying artificial intelligence",
    "models learn patterns from data",
    "neural networks have many layers",
    "training data is important for models",
    "testing helps evaluate performance",
    "optimization improves model accuracy",
    "algorithms solve complex problems",
    "feature engineering improves results",
    "big data requires efficient processing",
    "cloud computing supports scalability",
    "software development requires practice",
    "coding improves problem solving skills",
    "debugging helps fix errors",
    "practice makes a person better"
]

Tokenization

In [ ]:
tokenized_sentences = [sentence.split() for sentence in sentences]
print(tokenized_sentences[:3])

[['I', 'am', 'learning', 'machine', 'learning'], ['I', 'like', 'deep', 'learning', 'models'], ['I', 'am', 'playing', 'with', 'friends']]


Vocabulary

In [ ]:
words = [word for sentence in tokenized_sentences for word in sentence]
vocab = sorted(set(words))

word_to_index = {word: i for i, word in enumerate(vocab)}
index_to_word = {i: word for word, i in word_to_index.items()}

print("Vocabulary size:", len(vocab))
print("Sample vocab:", vocab[:10])

Vocabulary size: 83
Sample vocab: ['I', 'a', 'accuracy', 'algorithms', 'am', 'are', 'artificial', 'better', 'big', 'book']


Context Window and Target generation

In [4]:
context_size = 2

contexts = []
targets = []

for sentence in tokenized_sentences:
    for i in range(len(sentence) - context_size):
        context = sentence[i:i+context_size]
        target = sentence[i+context_size]

        contexts.append(context)
        targets.append(target)

Sliding Window

In [5]:
for i in range(10):
    print(f"{contexts[i]} ---> {targets[i]}")

['I', 'am'] ---> learning
['am', 'learning'] ---> machine
['learning', 'machine'] ---> learning
['I', 'like'] ---> deep
['like', 'deep'] ---> learning
['deep', 'learning'] ---> models
['I', 'am'] ---> playing
['am', 'playing'] ---> with
['playing', 'with'] ---> friends
['I', 'am'] ---> reading


Text to Index 

In [6]:
import numpy as np

X = [[word_to_index[word] for word in context] for context in contexts]
y = [word_to_index[word] for word in targets]

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (72, 2)
y shape: (72,)


Index to Embedding vectors

In [7]:
import torch
import torch.nn as nn

class NextWordModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, context_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.fc = nn.Linear(context_size * embedding_dim, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        x = x.view(x.shape[0], -1)
        out = self.fc(x)
        return out

Training

In [ ]:
vocab_size = len(vocab)
embedding_dim = 10

model = NextWordModel(vocab_size, embedding_dim, context_size)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

X_tensor = torch.tensor(X, dtype=torch.long)
y_tensor = torch.tensor(y, dtype=torch.long)

for epoch in range(100):
    optimizer.zero_grad()
    outputs = model(X_tensor)
    loss = loss_fn(outputs, y_tensor)
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item()}")

Epoch 0, Loss: 4.623752593994141
Epoch 10, Loss: 3.071373224258423
Epoch 20, Loss: 1.8002063035964966
Epoch 30, Loss: 0.9206774830818176
Epoch 40, Loss: 0.4691782593727112
Epoch 50, Loss: 0.28332090377807617
Epoch 60, Loss: 0.2128925919532776
Epoch 70, Loss: 0.18323472142219543
Epoch 80, Loss: 0.16850349307060242
Epoch 90, Loss: 0.16009755432605743


Prediction of next word

In [10]:
def predict_next(context_words):
    context_idx = torch.tensor([[word_to_index[word] for word in context_words]])
    output = model(context_idx)
    predicted_idx = torch.argmax(output).item()
    return index_to_word[predicted_idx]

Results

In [17]:
print(predict_next(["I", "am"]))
print(predict_next(["on", "a"]))
print(predict_next(["is", "widely"]))

reading
project
used


### LLM Failures

In [3]:
from langchain_groq.chat_models import ChatGroq
import os

Groq_Token = os.getenv("CHAT_GROQ_API_KEY")

llm = ChatGroq(
    api_key=Groq_Token,
    model="llama-3.3-70b-versatile",
    temperature=0.7
)

Hallucinations

In [ ]:
from langchain_core.messages import HumanMessage

prompt = """
You are a scientific literature expert.

Provide:
- authors
- affiliations
- DOI
- methodology (50 words)
- experimental results (50 words)

for the paper:
'Quantum Error Propagation in Recursive Biological Networks'
published in Nature Physics in 2024.
"""

response = llm.invoke([HumanMessage(content=prompt)])

print(response.content)

Authors: J. Zhang, M. Chen, Y. Wang
Affiliations: Harvard University, Stanford University, University of California
DOI: 10.1038/s41567-024-02341-9
Methodology: Theoretical modeling and simulations of quantum error propagation in recursive biological networks.
Experimental Results: Quantum error rates decreased by 30% in optimized recursive networks, outperforming classical counterparts.


- The Research paper doesnot exist yet the model fabricated the details. The output looks academically legitimate. That is what makes hallucinations dangerous in production systems.

Stale Knowledge

In [4]:
prompt = """
Who won the most recent IPL season?
"""

response = llm.invoke([HumanMessage(content=prompt)])

print(response.content)

The most recent IPL season was the 2022 Indian Premier League, which was won by the Gujarat Titans. They defeated the Rajasthan Royals in the final on May 29, 2022, at the Narendra Modi Stadium in Ahmedabad.


- The model treated 2022 as the latest IPL season even though multiple seasons have passed by. This is expected as the model was updated before the completion of the 2023 season.

Non Determinsim

In [47]:
from langchain_core.messages import HumanMessage
from langchain_groq import ChatGroq

llm = ChatGroq(
    api_key=Groq_Token,
    model="llama-3.3-70b-versatile",
    temperature=1.2
)

prompt = """
Pick exactly ONE candidate for an ML Engineer role and dont only consider their experience or only skills we need mix of both.

Candidate A: Python, TensorFlow, NLP, 5 years
Candidate B: Python, PyTorch, 6 years
Candidate C: Python, Computer Vision, 2 years

Return only the selected candidate name like "A".
"""

results = []

for i in range(15):
    response = llm.invoke([HumanMessage(content=prompt)])
    results.append(response.content.strip())

print(results)
print("\nUnique:", set(results))

['B', 'B', 'A', 'B', 'A', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B']

Unique: {'A', 'B'}


- The model showed non-deterministic candidate selection under ambiguous evaluation criteria, leading to inconsistent hiring decisions across identical runs.